# 🌿 TomLLB#1 — Qwen3-8B OpenAI-Compatible API

**Developer:** Md. Hassanul Hossain Tomal  
**Model:** Qwen3-8B  
**Hardware:** Google Colab T4 (16 GB VRAM)  
**API:** OpenAI-compatible `/v1`  
This notebook loads Qwen3-8B in 4-bit quantization and exposes it through a temporary Cloudflare Quick Tunnel.

**Important:** Keep this Colab runtime running while your agent uses the API. The Cloudflare URL is temporary and changes when a new tunnel is created.


In [ ]:
!nvidia-smi


## 📦 Install dependencies

The Colab-compatible `requests` version is pinned to avoid the dependency conflict with Google Colab.

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes fastapi uvicorn
!pip -q install requests==2.32.4
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared


## ⚙️ TomLLB#1 Configuration


In [ ]:
BRAND_NAME = "TomLLB#1"
DEVELOPER_NAME = "Md. Hassanul Hossain Tomal"

# Keep the real model name for OpenAI-compatible clients.
MODEL_NAME = "Qwen/Qwen3-8B"
MODEL_ID = "Qwen3-8B"

# API key requested for this branded deployment.
API_KEY = "hassanul.dev"
PORT = 8000

print("=" * 72)
print("🌿 " + BRAND_NAME)
print("=" * 72)
print("Developer :", DEVELOPER_NAME)
print("Model     :", MODEL_ID)
print("API Key   :", API_KEY)
print("Port      :", PORT)
print("=" * 72)


## 🧠 Load Qwen3-8B in 4-bit


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Select Runtime → Change runtime type → T4 GPU.")

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
)
model.eval()

print("\n✓ Qwen3-8B loaded successfully.")


## 🚀 Start OpenAI-compatible FastAPI server


In [ ]:
import threading
import time
import json
import uuid
from typing import List, Optional, Any

from fastapi import FastAPI, Header, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
import uvicorn

app = FastAPI(
    title=f"{BRAND_NAME} API",
    version="1.0.0",
    description=f"OpenAI-compatible API for {MODEL_ID}"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)

class Message(BaseModel):
    role: str
    content: Any

class ChatRequest(BaseModel):
    model: str = MODEL_ID
    messages: List[Message]
    max_tokens: Optional[int] = Field(default=512, ge=1, le=4096)
    temperature: Optional[float] = Field(default=0.7, ge=0.0, le=2.0)
    top_p: Optional[float] = Field(default=0.9, ge=0.0, le=1.0)
    stream: Optional[bool] = False

def check_key(authorization: Optional[str]):
    if not authorization or not authorization.startswith("Bearer "):
        raise HTTPException(status_code=401, detail="Missing Bearer API key")
    if authorization[7:] != API_KEY:
        raise HTTPException(status_code=401, detail="Invalid API key")

def content_to_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and item.get("type") == "text":
                parts.append(str(item.get("text", "")))
        return "\n".join(parts)
    return str(content)

def prepare_inputs(messages):
    clean_messages = [
        {"role": m.role, "content": content_to_text(m.content)}
        for m in messages
        if m.role in ("system", "user", "assistant")
    ]

    if not clean_messages:
        raise HTTPException(status_code=400, detail="messages cannot be empty")

    prompt = tokenizer.apply_chat_template(
        clean_messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    return tokenizer([prompt], return_tensors="pt").to(model.device)

def generate(messages, max_tokens, temperature, top_p):
    inputs = prepare_inputs(messages)

    generation_kwargs = {
        "max_new_tokens": max_tokens,
        "top_p": top_p,
        "do_sample": temperature > 0,
        "pad_token_id": tokenizer.eos_token_id,
    }

    if temperature > 0:
        generation_kwargs["temperature"] = temperature

    with torch.no_grad():
        outputs = model.generate(**inputs, **generation_kwargs)

    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

def make_request_id():
    return "chatcmpl-tomllb1-" + uuid.uuid4().hex[:12]

def stream_answer(answer, request_id):
    chunk_size = 40

    for i in range(0, len(answer), chunk_size):
        piece = answer[i:i + chunk_size]
        chunk = {
            "id": request_id,
            "object": "chat.completion.chunk",
            "created": int(time.time()),
            "model": MODEL_ID,
            "choices": [{
                "index": 0,
                "delta": {"content": piece},
                "finish_reason": None
            }]
        }
        yield "data: " + json.dumps(chunk, ensure_ascii=False) + "\n\n"

    final_chunk = {
        "id": request_id,
        "object": "chat.completion.chunk",
        "created": int(time.time()),
        "model": MODEL_ID,
        "choices": [{
            "index": 0,
            "delta": {},
            "finish_reason": "stop"
        }]
    }

    yield "data: " + json.dumps(final_chunk) + "\n\n"
    yield "data: [DONE]\n\n"

@app.get("/")
def root():
    return {
        "brand": BRAND_NAME,
        "developer": DEVELOPER_NAME,
        "model": MODEL_ID,
        "api": "OpenAI-compatible",
        "status": "online"
    }

@app.get("/health")
def health():
    return {
        "status": "ok",
        "brand": BRAND_NAME,
        "model": MODEL_ID
    }

@app.get("/v1/models")
def models(authorization: Optional[str] = Header(default=None)):
    check_key(authorization)
    return {
        "object": "list",
        "data": [{
            "id": MODEL_ID,
            "object": "model",
            "created": int(time.time()),
            "owned_by": BRAND_NAME
        }]
    }

@app.post("/v1/chat/completions")
def chat_completions(
    req: ChatRequest,
    authorization: Optional[str] = Header(default=None)
):
    check_key(authorization)

    if req.model not in (MODEL_ID, MODEL_NAME):
        raise HTTPException(status_code=404, detail=f"Unknown model: {req.model}")

    request_id = make_request_id()
    answer = generate(req.messages, req.max_tokens, req.temperature, req.top_p)

    if req.stream:
        return StreamingResponse(
            stream_answer(answer, request_id),
            media_type="text/event-stream",
            headers={
                "Cache-Control": "no-cache",
                "Connection": "keep-alive",
                "X-Accel-Buffering": "no"
            }
        )

    return {
        "id": request_id,
        "object": "chat.completion",
        "created": int(time.time()),
        "model": MODEL_ID,
        "choices": [{
            "index": 0,
            "message": {
                "role": "assistant",
                "content": answer
            },
            "finish_reason": "stop"
        }],
        "usage": {
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "total_tokens": 0
        }
    }

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)

print("=" * 72)
print(f"✓ 🌿 {BRAND_NAME} API server started")
print(f"  Local endpoint: http://127.0.0.1:{PORT}")
print("=" * 72)


## 🌐 Start Cloudflare Quick Tunnel + automatic connectivity test

This cell intentionally creates **only one** tunnel and waits/retries before testing DNS. If the tunnel dies, simply re-run this cell.

In [ ]:
import subprocess
import time
import re
import requests

# Stop an old tunnel if this cell is being re-run.
try:
    cloudflare.terminate()
    time.sleep(2)
except Exception:
    pass

print("Starting Cloudflare Quick Tunnel...")

cloudflare = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url",
        f"http://127.0.0.1:{PORT}",
        "--no-autoupdate",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

public_url = None
deadline = time.time() + 60

while time.time() < deadline:
    line = cloudflare.stdout.readline()

    if line:
        print(line.strip())

        match = re.search(
            r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com",
            line
        )

        if match:
            public_url = match.group(0)
            break
    else:
        time.sleep(0.2)

if not public_url:
    raise RuntimeError(
        "Cloudflare did not provide a public URL. Re-run this cell."
    )

base_url = public_url + "/v1"

print("\n" + "=" * 72)
print(f"🌿 {BRAND_NAME} API IS READY")
print("=" * 72)
print("Developer :", DEVELOPER_NAME)
print("Model     :", MODEL_ID)
print("Base URL  :", base_url)
print("API Key   :", API_KEY)
print("=" * 72)

headers = {"Authorization": f"Bearer {API_KEY}"}

# Allow time for Quick Tunnel DNS propagation.
print("\nWaiting for Cloudflare DNS propagation...")
time.sleep(8)

models_ok = False

for attempt in range(1, 11):
    try:
        print(f"Testing /v1/models... attempt {attempt}/10")

        r = requests.get(
            base_url + "/models",
            headers=headers,
            timeout=15,
        )

        print("Status:", r.status_code)
        print(r.text)

        if r.status_code == 200:
            models_ok = True
            break

    except requests.RequestException as e:
        print("Connection error:", e)

    time.sleep(3)

if not models_ok:
    raise RuntimeError(
        "Could not reach the Cloudflare tunnel. Re-run this cell to create a fresh tunnel."
    )

print("\nTesting /v1/chat/completions...")

payload = {
    "model": MODEL_ID,
    "messages": [
        {
            "role": "user",
            "content": "Reply with exactly: TOMLLB API IS WORKING"
        }
    ],
    "max_tokens": 50,
    "temperature": 0.2,
    "stream": False,
}

r = requests.post(
    base_url + "/chat/completions",
    headers=headers,
    json=payload,
    timeout=180,
)

print("Status:", r.status_code)
print(r.text)

if r.status_code == 200:
    print("\n" + "=" * 72)
    print(f"✓ 🌿 {BRAND_NAME} API IS FULLY WORKING")
    print("=" * 72)
else:
    raise RuntimeError("Chat endpoint test failed. Check the response above.")


## 🔌 Connect your agent

Use the **exact Base URL printed above**:

```text
Base URL:  https://YOUR-URL.trycloudflare.com/v1
API Key:   hassanul.dev
Model:     Qwen3-8B
```

Do **not** add `/chat/completions` to the Base URL.

### OpenAI-compatible Python client

```python
from openai import OpenAI

client = OpenAI(
    base_url="https://YOUR-URL.trycloudflare.com/v1",
    api_key="hassanul.dev",
)

response = client.chat.completions.create(
    model="Qwen3-8B",
    messages=[{"role": "user", "content": "Hello!"}],
)

print(response.choices[0].message.content)
```


## 🧪 Ubuntu cURL test

Replace `YOUR-URL` with the exact URL printed by the tunnel cell.

### Models

```bash
curl -i "https://YOUR-URL.trycloudflare.com/v1/models" \\
  -H "Authorization: Bearer hassanul.dev"
```

### Chat

```bash
curl -i -X POST "https://YOUR-URL.trycloudflare.com/v1/chat/completions" \\
  -H "Content-Type: application/json" \\
  -H "Authorization: Bearer hassanul.dev" \\
  -d '{
    "model": "Qwen3-8B",
    "messages": [
      {"role": "user", "content": "Hello from Ubuntu!"}
    ],
    "max_tokens": 200,
    "temperature": 0.7,
    "stream": false
  }'
```

### Streaming test

```bash
curl -N -X POST "https://YOUR-URL.trycloudflare.com/v1/chat/completions" \\
  -H "Content-Type: application/json" \\
  -H "Authorization: Bearer hassanul.dev" \\
  -d '{
    "model": "Qwen3-8B",
    "messages": [
      {"role": "user", "content": "Tell me a short story."}
    ],
    "max_tokens": 200,
    "stream": true
  }'
```


## ⚠️ Runtime / tunnel notes

- Keep this Colab runtime running while your agent uses the API.
- The Cloudflare Quick Tunnel URL is temporary.
- Restarting the tunnel/runtime can produce a new URL.
- If `/v1/models` reports a DNS/NameResolution error, re-run the **Cloudflare Quick Tunnel + automatic connectivity test** cell.
- Only run one Cloudflare tunnel cell at a time.
- The API supports both `stream: false` and OpenAI-style `stream: true` responses.

### 🌿 TomLLB#1
**Md. Hassanul Hossain Tomal · Qwen3-8B · Google Colab T4**
